# Sprint 0 — 01: Introduction and data

## The brief

A literature review of the resources for doing data science in our chosen
domain: find books, websites and other resources that provide code, run their
examples, modify them to make our own visualisations of the data they analyse,
and combine the best into a report with a logical structure.
([Assessment 0](https://dsbristol.github.io/dst/assets/assessments/Assessment0.pdf))

## The domain

<What the domain is, and why we chose it. It carries into Sprint 1, which needs
a supervised problem — so say what LABELLED data exists here.>

## The team

Oscar, Alex and Louis. <Who is doing what, and the agreed equity.>

## Setup

Paths are built with `pathlib` from this notebook's location, so the report
runs the same on everyone's machine (the Python equivalent of the example
project's use of `fs` in R). The layout follows the unit's convention:

* `data/raw/` — downloaded once, never edited
* `data/processed/` — regenerated by the code, not committed
* `data/output/` — figures and tables the report keeps

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent            # notebooks run from report/
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
OUTPUT = ROOT / "data" / "output"
for d in (RAW, PROCESSED, OUTPUT):
    d.mkdir(parents=True, exist_ok=True)


def fetch(url: str, name: str) -> Path:
    """Download `url` into data/raw/ once; later runs reuse the copy."""
    path = RAW / name
    if not path.exists():
        urlretrieve(url, path)
    return path

print("project root:", ROOT)

## Getting the data

We use two public datasets from the UCI Machine Learning Repository. Both are
downloaded into `data/raw/` the first time this notebook runs, and are never
edited there: anything we change is done in code, so the path from the
original file to every figure can be followed.

| Dataset | Rows | Label | Source |
|---|---|---|---|
| Default of Credit Card Clients (Taiwan) | 30,000 | default next month | Yeh (2009), [UCI 350](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients), doi:10.24432/C55S3H |
| Statlog German Credit | 1,000 | good / bad loan | Hofmann (1994), [UCI 144](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data), doi:10.24432/C5NC77 |

Reading the Taiwan file needs the `xlrd` package, because it is an old-format
Excel `.xls` (it is in `requirements.txt`).

In [ ]:
from zipfile import ZipFile

UCI = "https://archive.ics.uci.edu/static/public"
SOURCES = {
    # folder in data/raw/   : (zip file on UCI,                            a file the zip contains)
    "taiwan-credit-default": (f"{UCI}/350/default+of+credit+card+clients.zip", "default of credit card clients.xls"),
    "german-credit":         (f"{UCI}/144/statlog+german+credit+data.zip",     "german.data"),
}


def download_raw(folder: str) -> Path:
    """Download and unzip one dataset into data/raw/<folder>/, unless it is already there."""
    url, expected = SOURCES[folder]
    target = RAW / folder
    if not (target / expected).exists():
        target.mkdir(parents=True, exist_ok=True)
        zip_path, _ = urlretrieve(url)          # to a temporary file
        with ZipFile(zip_path) as z:
            z.extractall(target)
    return target


TAIWAN_DIR = download_raw("taiwan-credit-default")
GERMAN_DIR = download_raw("german-credit")
print(sorted(p.name for p in TAIWAN_DIR.iterdir()))
print(sorted(p.name for p in GERMAN_DIR.iterdir()))

### Taiwan credit-card clients

One row per cardholder, from a Taiwanese bank in 2005. The spreadsheet has a
title row above the real column names, so we skip it (`header=1`), and use the
`ID` column as the row label. The label is `default payment next month`
(1 = defaulted). The columns are described on the UCI page: `LIMIT_BAL` is
the credit limit; `PAY_0`, `PAY_2`…`PAY_6` are repayment status for the last
six months (note there is no `PAY_1`); `BILL_AMT*` and `PAY_AMT*` are the bill
and payment amounts for the same months.

In [ ]:
taiwan = pd.read_excel(TAIWAN_DIR / "default of credit card clients.xls",
                       header=1, index_col="ID")

print(taiwan.shape)                                             # expect (30000, 24)
print(taiwan["default payment next month"].value_counts())    # 1 = default
taiwan.head()

### German Credit

One row per loan applicant. The file has no header row and is separated by
spaces; the column meanings come from the dataset's own codebook,
`data/raw/german-credit/german.doc`. Categorical columns hold codes such as
`A11` (checking account below 0 DM), also explained there. The label is
`class`: **1 = good, 2 = bad**. The codebook also gives the cost matrix we use
later: calling a bad loan good costs 5, calling a good loan bad costs 1.

In [ ]:
GERMAN_COLUMNS = [
    "checking_account", "duration_months", "credit_history", "purpose",
    "credit_amount", "savings", "employment_since", "installment_rate",
    "personal_status_sex", "other_debtors", "residence_since", "property",
    "age", "other_installment_plans", "housing", "existing_credits", "job",
    "people_liable", "telephone", "foreign_worker", "class",
]   # in the order of german.doc, attributes 1-20, then the label

german = pd.read_csv(GERMAN_DIR / "german.data", sep=" ", header=None,
                     names=GERMAN_COLUMNS)

print(german.shape)                    # expect (1000, 21)
print(german["class"].value_counts())  # 1 = good, 2 = bad
german.head()

In [ ]:
# A check that both files were read the way we expect. If a download changes
# or a read goes wrong, this stops the notebook here rather than later.
assert taiwan.shape == (30000, 24), taiwan.shape
assert german.shape == (1000, 21), german.shape
assert set(taiwan["default payment next month"]) == {0, 1}
assert set(german["class"]) == {1, 2}
print("Both datasets loaded as expected.")

## Resources found

| Resource | Type | Language | What it does | Ran it? | Link |
|---|---|---|---|---|---|
| | book / site / repo / paper | | | | |